
# Roxy notebook example: Normalized positional descriptors

This notebook is a **reference implementation example** for the **normalized positional descriptor family** in Roxy.

These descriptors summarize **where residues or residue groups tend to appear along the sequence**, using positions normalized by sequence length. This is useful because many sequences show positional biases even when their global composition is similar.

## Covered outputs

This notebook implements examples such as:

- first occurrence position
- last occurrence position
- mean normalized position
- median normalized position
- positional standard deviation
- positional span
- N-terminal bias and C-terminal bias proxies
- center-of-mass position for residue groups
- normalized positional summaries for individual residues and grouped residues
- class-style implementation for later migration into Roxy

The notebook is designed as a **clean teaching implementation** so it can later become part of the real Roxy package.


In [1]:

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "pos_1",
            "pos_2",
            "pos_3",
            "pos_4",
            "pos_5",
            "pos_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,pos_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,pos_2,GGGGGGGGGGGGGGG,B
2,pos_3,KRRKRRKRRKRRDDDDEE,A
3,pos_4,ACDEFGHIKLMNPQRSTVWY,B
4,pos_5,PPPPGSSSSSTTTTNNQQQ,A
5,pos_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [3]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

AA_GROUPS = {
    "positive": set("KRH"),
    "negative": set("DE"),
    "charged": set("KRHDE"),
    "polar": set("STNQCYWHKRDE"),
    "nonpolar": set("AVLIMFGP"),
    "aromatic": set("FWYH"),
    "aliphatic": set("AVLIM"),
    "hydrophobic": set("AVLIMFWCY"),
    "hydrophilic": set("RNDQEHKST"),
    "disorder_promoting": set("ARGQSEPK"),
    "order_promoting": set("CWYFILNV"),
}

AA_SINGLETS = {
    "K": set("K"),
    "R": set("R"),
    "D": set("D"),
    "E": set("E"),
    "G": set("G"),
    "P": set("P"),
    "W": set("W"),
    "Y": set("Y"),
}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def positions_of_group(seq: str, aa_group):
    return [i + 1 for i, aa in enumerate(seq) if aa in aa_group]


def normalize_positions(positions, seq_len: int):
    if seq_len == 0 or len(positions) == 0:
        return []
    return [p / seq_len for p in positions]


def positional_summary(positions, seq_len: int):
    norm_positions = normalize_positions(positions, seq_len)

    if len(norm_positions) == 0:
        return {
            "count": 0,
            "first_norm": np.nan,
            "last_norm": np.nan,
            "mean_norm": np.nan,
            "median_norm": np.nan,
            "std_norm": np.nan,
            "span_norm": np.nan,
            "n_bias": np.nan,
            "c_bias": np.nan,
            "center_mass_norm": np.nan,
        }

    first_norm = float(norm_positions[0])
    last_norm = float(norm_positions[-1])
    mean_norm = float(np.mean(norm_positions))
    median_norm = float(np.median(norm_positions))
    std_norm = float(np.std(norm_positions, ddof=0))
    span_norm = float(last_norm - first_norm)
    center_mass_norm = mean_norm

    # Simple positional bias proxies
    n_bias = float(np.mean(np.array(norm_positions) <= 0.33))
    c_bias = float(np.mean(np.array(norm_positions) >= 0.67))

    return {
        "count": len(norm_positions),
        "first_norm": first_norm,
        "last_norm": last_norm,
        "mean_norm": mean_norm,
        "median_norm": median_norm,
        "std_norm": std_norm,
        "span_norm": span_norm,
        "n_bias": n_bias,
        "c_bias": c_bias,
        "center_mass_norm": center_mass_norm,
    }


## Core descriptor function

In [5]:

def positional_descriptors(seq: str) -> dict:
    seq = clean_sequence(seq)
    seq_len = len(seq)

    out = {
        "pos_length": seq_len,
        "pos_valid_residue_count": seq_len,
    }

    if seq_len == 0:
        return out

    tracked_groups = {
        "charged": AA_GROUPS["charged"],
        "hydrophobic": AA_GROUPS["hydrophobic"],
        "aromatic": AA_GROUPS["aromatic"],
        "polar": AA_GROUPS["polar"],
        "positive": AA_GROUPS["positive"],
        "negative": AA_GROUPS["negative"],
        "gly": AA_SINGLETS["G"],
        "pro": AA_SINGLETS["P"],
        "trp": AA_SINGLETS["W"],
        "tyr": AA_SINGLETS["Y"],
    }

    for name, group in tracked_groups.items():
        positions = positions_of_group(seq, group)
        summary = positional_summary(positions, seq_len)

        out[f"pos_{name}_count"] = summary["count"]
        out[f"pos_{name}_first_norm"] = summary["first_norm"]
        out[f"pos_{name}_last_norm"] = summary["last_norm"]
        out[f"pos_{name}_mean_norm"] = summary["mean_norm"]
        out[f"pos_{name}_median_norm"] = summary["median_norm"]
        out[f"pos_{name}_std_norm"] = summary["std_norm"]
        out[f"pos_{name}_span_norm"] = summary["span_norm"]
        out[f"pos_{name}_n_bias"] = summary["n_bias"]
        out[f"pos_{name}_c_bias"] = summary["c_bias"]
        out[f"pos_{name}_center_mass_norm"] = summary["center_mass_norm"]

    return out


## Functional usage on one sequence

In [6]:

example = positional_descriptors(df_demo.loc[0, "sequence"])
list(example.items())[:18]


[('pos_length', 24),
 ('pos_valid_residue_count', 24),
 ('pos_charged_count', 4),
 ('pos_charged_first_norm', 0.08333333333333333),
 ('pos_charged_last_norm', 1.0),
 ('pos_charged_mean_norm', 0.7083333333333334),
 ('pos_charged_median_norm', 0.875),
 ('pos_charged_std_norm', 0.3691676072222781),
 ('pos_charged_span_norm', 0.9166666666666666),
 ('pos_charged_n_bias', 0.25),
 ('pos_charged_c_bias', 0.75),
 ('pos_charged_center_mass_norm', 0.7083333333333334),
 ('pos_hydrophobic_count', 14),
 ('pos_hydrophobic_first_norm', 0.041666666666666664),
 ('pos_hydrophobic_last_norm', 0.9166666666666666),
 ('pos_hydrophobic_mean_norm', 0.4523809523809524),
 ('pos_hydrophobic_median_norm', 0.4375),
 ('pos_hydrophobic_std_norm', 0.2601401593263352)]

## Apply positional descriptors to the full dataset

In [7]:

df_pos = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(positional_descriptors).apply(pd.Series),
    ],
    axis=1,
)

df_pos.head()


,sequence_id,sequence,label,pos_length,pos_valid_residue_count,pos_charged_count,pos_charged_first_norm,pos_charged_last_norm,pos_charged_mean_norm,pos_charged_median_norm,...,pos_tyr_count,pos_tyr_first_norm,pos_tyr_last_norm,pos_tyr_mean_norm,pos_tyr_median_norm,pos_tyr_std_norm,pos_tyr_span_norm,pos_tyr_n_bias,pos_tyr_c_bias,pos_tyr_center_mass_norm
0,pos_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,4.0,0.083333,1.00,0.708333,0.875000,...,1.0,0.708333,0.708333,0.708333,0.708333,0.0,0.0,0.0,1.0,0.708333
1,pos_2,GGGGGGGGGGGGGGG,B,15.0,15.0,0.0,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,pos_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,18.0,0.055556,1.00,0.527778,0.527778,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,pos_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,5.0,0.150000,0.75,0.380000,0.350000,...,1.0,1.000000,1.000000,1.000000,1.000000,0.0,0.0,0.0,1.0,1.000000
4,pos_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,0.0,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Inspect positional descriptor columns

In [8]:

pos_cols = [c for c in df_pos.columns if c.startswith("pos_") and c not in {"pos_length", "pos_valid_residue_count"}]
len(pos_cols), pos_cols[:16]


(100,
 ['pos_charged_count',
  'pos_charged_first_norm',
  'pos_charged_last_norm',
  'pos_charged_mean_norm',
  'pos_charged_median_norm',
  'pos_charged_std_norm',
  'pos_charged_span_norm',
  'pos_charged_n_bias',
  'pos_charged_c_bias',
  'pos_charged_center_mass_norm',
  'pos_hydrophobic_count',
  'pos_hydrophobic_first_norm',
  'pos_hydrophobic_last_norm',
  'pos_hydrophobic_mean_norm',
  'pos_hydrophobic_median_norm',
  'pos_hydrophobic_std_norm'])

In [9]:

df_pos[
    [
        "sequence_id",
        "pos_charged_first_norm",
        "pos_charged_last_norm",
        "pos_charged_mean_norm",
        "pos_hydrophobic_mean_norm",
        "pos_aromatic_center_mass_norm",
        "pos_gly_n_bias",
        "pos_pro_c_bias",
    ]
]


,sequence_id,pos_charged_first_norm,pos_charged_last_norm,pos_charged_mean_norm,pos_hydrophobic_mean_norm,pos_aromatic_center_mass_norm,pos_gly_n_bias,pos_pro_c_bias
0,pos_1,0.083333,1.000000,0.708333,0.452381,0.5000,0.000000,NaN
1,pos_2,NaN,NaN,NaN,NaN,NaN,0.266667,NaN
2,pos_3,0.055556,1.000000,0.527778,NaN,NaN,NaN,NaN
3,pos_4,0.150000,0.750000,0.380000,0.522222,0.6375,1.000000,0.0
4,pos_5,NaN,NaN,NaN,NaN,NaN,1.000000,0.0
5,pos_6,0.285714,0.904762,0.619048,0.650794,NaN,0.000000,0.0


## Dataset-level summary

In [10]:

pos_summary = (
    df_pos[pos_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

pos_summary.head(15)


,descriptor,mean_value
0,pos_polar_count,11.166667
1,pos_charged_count,5.500000
2,pos_hydrophobic_count,4.833333
3,pos_positive_count,3.833333
4,pos_gly_count,3.166667
5,pos_negative_count,1.666667
6,pos_aromatic_count,1.666667
7,pos_pro_count,1.166667
8,pos_tyr_c_bias,1.000000
9,pos_polar_last_norm,0.980952


## Sanity checks

In [11]:

assert "pos_charged_first_norm" in df_pos.columns
assert "pos_hydrophobic_mean_norm" in df_pos.columns
assert "pos_aromatic_span_norm" in df_pos.columns
assert "pos_positive_n_bias" in df_pos.columns
assert "pos_negative_c_bias" in df_pos.columns
assert df_pos["pos_length"].min() > 0

print(f"Number of normalized positional descriptor columns: {len(pos_cols)}")
print("Normalized positional descriptor checks passed.")


Number of normalized positional descriptor columns: 100
Normalized positional descriptor checks passed.


## Class-style implementation closer to the real package

In [12]:

class NormalizedPositionalDescriptors:
    """Example class-style positional implementation for later migration into Roxy."""

    def transform_sequence(self, seq: str) -> dict:
        return positional_descriptors(seq)

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


pos_transformer = NormalizedPositionalDescriptors()
pos_matrix = pos_transformer.transform(df_demo["sequence"].tolist())
pos_matrix.head()


,pos_length,pos_valid_residue_count,pos_charged_count,pos_charged_first_norm,pos_charged_last_norm,pos_charged_mean_norm,pos_charged_median_norm,pos_charged_std_norm,pos_charged_span_norm,pos_charged_n_bias,...,pos_tyr_count,pos_tyr_first_norm,pos_tyr_last_norm,pos_tyr_mean_norm,pos_tyr_median_norm,pos_tyr_std_norm,pos_tyr_span_norm,pos_tyr_n_bias,pos_tyr_c_bias,pos_tyr_center_mass_norm
0,24,24,4,0.083333,1.00,0.708333,0.875000,0.369168,0.916667,0.250000,...,1,0.708333,0.708333,0.708333,0.708333,0.0,0.0,0.0,1.0,0.708333
1,15,15,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,18,18,18,0.055556,1.00,0.527778,0.527778,0.288229,0.944444,0.277778,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20,20,5,0.150000,0.75,0.380000,0.350000,0.213542,0.600000,0.400000,...,1,1.000000,1.000000,1.000000,1.000000,0.0,0.0,0.0,1.0,1.000000
4,19,19,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Merge transformer output back to the dataset

In [13]:

df_pos_class = pd.concat([df_demo, pos_matrix], axis=1)
df_pos_class.head()


,sequence_id,sequence,label,pos_length,pos_valid_residue_count,pos_charged_count,pos_charged_first_norm,pos_charged_last_norm,pos_charged_mean_norm,pos_charged_median_norm,...,pos_tyr_count,pos_tyr_first_norm,pos_tyr_last_norm,pos_tyr_mean_norm,pos_tyr_median_norm,pos_tyr_std_norm,pos_tyr_span_norm,pos_tyr_n_bias,pos_tyr_c_bias,pos_tyr_center_mass_norm
0,pos_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,4,0.083333,1.00,0.708333,0.875000,...,1,0.708333,0.708333,0.708333,0.708333,0.0,0.0,0.0,1.0,0.708333
1,pos_2,GGGGGGGGGGGGGGG,B,15,15,0,NaN,NaN,NaN,NaN,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,pos_3,KRRKRRKRRKRRDDDDEE,A,18,18,18,0.055556,1.00,0.527778,0.527778,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,pos_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,5,0.150000,0.75,0.380000,0.350000,...,1,1.000000,1.000000,1.000000,1.000000,0.0,0.0,0.0,1.0,1.000000
4,pos_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,0,NaN,NaN,NaN,NaN,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move helper logic into `roxy/sequence/position.py`
- keep residue groups in `roxy/core/constants.py`
- expose a class such as `NormalizedPositionalDescriptors`
- allow configurable:
  - tracked groups
  - residue-specific positional summaries
  - positional bias thresholds
- add tests for:
  - empty sequences
  - groups absent from the sequence
  - strong N-terminal enrichment
  - strong C-terminal enrichment
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [14]:
# df_pos.to_csv("demo_normalized_positional_descriptors.csv", index=False)
